# 파티 코치 LoRA 파인튜닝 — 집 PC · RTX 3060 12GB

Qwen3.5-4B + Unsloth. **bf16 LoRA (약 10GB), 최대 길이 4096토큰**.
같은 내용의 다른 버전: `finetune_laptop`(RTX 4050) · `finetune_home`(RTX 3060) · `finetune_colab`(T4). 설정 셀만 다르다.

`llm/agent`의 **시스템 프롬프트·도구 정의를 그대로** 써서, 모델이 도구 호출 방식과 파티 판단을 배우게 한다.
사실(종족값·사용률 등)은 외우게 하지 않는다. 학습 데이터 속 도구 결과는 항상 실제 back API 값이어야 한다.

> 메모리가 모자라면 (OOM) `LOAD_IN_4BIT = True`로 바꾼다.

**준비**
- WSL2(Ubuntu)에서 venv를 따로 만들고 `pip install unsloth jupyter httpx` (Windows의 `.venv`와 섞지 않는다)
- 저장소의 `llm/training/`에서 노트북을 연다. `llm/agent/prompt.py`, `tools.py`는 자동으로 찾는다.
- 학습 데이터는 같은 폴더의 `train.jsonl` (git 제외)

`train.jsonl`이 없으면 아래 예시 대화 2개를 반복해서 **파이프라인만 확인**한다 (실제 학습용 아님).

In [ ]:
import os, sys, json, re, subprocess
from pathlib import Path

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])

## 1. 설치
WSL2 venv에 `pip install unsloth jupyter httpx`를 한 번 해 두면 된다 (이 노트북에서는 설치하지 않음).

## 2. 설정
이 버전의 GPU에 맞춘 값이다. 다른 GPU에서 열었으면 경고가 뜬다.

In [ ]:
import torch

MODEL_NAME = 'unsloth/Qwen3.5-4B'          # 원본: Qwen/Qwen3.5-4B (Apache 2.0)
LOAD_IN_4BIT = False
MAX_SEQ_LEN = 4096                     # 넘는 대화는 학습에서 뺀다 (6번 셀)
BF16 = torch.cuda.is_bf16_supported()     # T4는 False → fp16

LORA_R = 16
EPOCHS = 2
LR = 2e-4
BATCH, GRAD_ACC = 1, 8                     # 실제 배치 = BATCH × GRAD_ACC

OUT_DIR = Path('outputs')                  # llm/training/outputs (git 제외)

gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 2**30
if 'RTX 3060' not in gpu.name:
    print(f'⚠ 이 노트북은 RTX 3060용인데 지금 GPU는 {gpu.name} ({VRAM_GB:.1f}GB). 맞는 버전을 쓰거나 위 값을 고칠 것')
print(f'{gpu.name} {VRAM_GB:.1f}GB | 4bit={LOAD_IN_4BIT} bf16={BF16} seq={MAX_SEQ_LEN} out={OUT_DIR}')

## 3. 시스템 프롬프트·도구 불러오기
서버(`llm/agent`)와 같은 프롬프트·도구로 학습해야 붙였을 때 그대로 동작한다.
도구 정의는 Claude 형식(`input_schema`)이라 Qwen 형식(`function.parameters`)으로 바꾼다.

In [ ]:
import importlib.util

def find_agent_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'llm/agent/tools.py').exists():
            return base / 'llm/agent'
    raise FileNotFoundError('저장소 안(llm/training/)에서 노트북을 열 것')

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

AGENT = find_agent_dir()
prompt = load('prompt', AGENT / 'prompt.py')
tools_mod = load('tools', AGENT / 'tools.py')        # httpx 필요 (Colab엔 기본 설치)

SYSTEM = prompt.SYSTEM
TOOLS = [{'type': 'function', 'function': {'name': t['name'], 'description': t['description'],
                                           'parameters': t['input_schema']}} for t in tools_mod.TOOLS]
print(AGENT, '| 도구', len(TOOLS), '개:', ', '.join(t['function']['name'] for t in TOOLS))

## 4. 학습 데이터
`train.jsonl` 한 줄 = 대화 하나. `system`은 비워 두면 위 SYSTEM을 넣는다.
```json
{"messages": [
  {"role": "user", "content": "[화면] 포맷: 더블 (doubles), ...\n\n한카리아스 어떻게 키워?"},
  {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": "search_pokemon", "arguments": {"query": "한카리아스"}}}]},
  {"role": "tool", "content": "<도구 결과 JSON (실제 back API 값)>"},
  {"role": "assistant", "content": "최종 답변"}
]}
```
- 사용자 메시지 앞의 `[화면]` 부분은 `prompt.screen_context()`로 만든다 (서버와 같게).
- 도구 결과는 **반드시 back API를 실제로 불러 얻은 값**을 넣는다. 지어낸 수치로 학습하면 모델이 수치를 지어내는 법을 배운다.
- 아래 예시의 도구 결과는 2026-10 DB 값을 줄여서 넣은 것이다.

In [ ]:
def ctx(fmt='doubles', party=None):
    return prompt.screen_context(fmt, party or [None] * 6)

def call(name, **args):
    return {'type': 'function', 'function': {'name': name, 'arguments': args}}

def result(obj):
    return {'role': 'tool', 'content': json.dumps(obj, ensure_ascii=False)}

DEMO = [
    # 1) 이름 → ID 찾기 → 상세(사용률) → 짧은 답
    {'messages': [
        {'role': 'user', 'content': ctx() + '\n\n한카리아스 더블에서 보통 어떻게 키워?'},
        {'role': 'assistant', 'content': '', 'tool_calls': [call('search_pokemon', query='한카리아스')]},
        result([{'id': 'garchomp', 'name_ko': '한카리아스', 'types': ['Dragon', 'Ground'],
                 'stats': {'hp': 108, 'atk': 130, 'def': 95, 'spa': 80, 'spd': 85, 'spe': 102}, 'rank': 9,
                 'megas': [{'id': 'garchompmega', 'name_ko': '메가한카리아스', 'stone': 'garchompite'},
                           {'id': 'garchompmegaz', 'name_ko': '메가한카리아스Z', 'stone': 'garchompitez'}]}]),
        {'role': 'assistant', 'content': '', 'tool_calls': [call('get_pokemon', id='garchomp', format='doubles')]},
        result({'id': 'garchomp', 'name_ko': '한카리아스', 'format': 'doubles', 'usage': {
            'rank': 9,
            'move': [{'id': 'protect', 'name_ko': '방어', 'pct': 80.5}, {'id': 'rockslide', 'name_ko': '스톤샤워', 'pct': 43.9},
                     {'id': 'dragonclaw', 'name_ko': '드래곤클로', 'pct': 43.0}, {'id': 'earthquake', 'name_ko': '지진', 'pct': 40.9}],
            'item': [{'id': 'garchompitez', 'name_ko': '한카리아스나이트Z', 'pct': 46.3},
                     {'id': 'lifeorb', 'name_ko': '생명의구슬', 'pct': 26.1},
                     {'id': 'choicescarf', 'name_ko': '구애스카프', 'pct': 9.4}],
            'ability': [{'id': 'roughskin', 'name_ko': '까칠한피부', 'pct': 96.7}],
            'nature': [{'id': 'modest', 'name_ko': '조심', 'pct': 35.6}, {'id': 'jolly', 'name_ko': '명랑', 'pct': 30.3},
                       {'id': 'adamant', 'name_ko': '고집', 'pct': 20.2}],
            'spread': [{'sp': 'hp2 atk32 spe32', 'pct': 24.6}, {'sp': 'hp2 spa32 spe32', 'pct': 17.9}]}}),
        {'role': 'assistant', 'content':
            '더블 픽률 9위이고, 절반 가까이(46.3%)가 **한카리아스나이트Z**를 들고 메가진화해요.\n'
            '- 기술: 방어(80.5%), 스톤샤워(43.9%), 드래곤클로(43.0%), 지진(40.9%)\n'
            '- 성격: 조심 35.6% / 명랑 30.3% — 특수형과 물리형으로 갈려요\n'
            '- SP: hp2 atk32 spe32(24.6%) 또는 hp2 spa32 spe32(17.9%)\n'
            '메가 대신 생명의구슬(26.1%)도 많이 써요. 육성형을 정해 드릴까요?'},
    ]},
    # 2) 규칙 질문은 시스템 프롬프트로 답하고 도구를 부르지 않음
    {'messages': [
        {'role': 'user', 'content': ctx('singles') + '\n\nSP는 최대 얼마까지 줄 수 있어?'},
        {'role': 'assistant', 'content': '능력치 하나에 0~32, 여섯 능력치 합계는 66까지예요. '
                                         '보통 두 능력치에 32씩 주고 남은 2를 다른 곳에 넣어요.'},
    ]},
]

TRAIN_FILE = next((p for p in ['train.jsonl', '/content/train.jsonl'] if Path(p).exists()), None)
if TRAIN_FILE:
    rows = [json.loads(l) for l in open(TRAIN_FILE, encoding='utf-8') if l.strip()]
    print(f'{TRAIN_FILE}: {len(rows)}개')
else:
    rows = DEMO * 20
    print('train.jsonl 없음 → 예시 2개 × 20 (파이프라인 확인용)')

for r in rows:
    if r['messages'][0]['role'] != 'system':
        r['messages'].insert(0, {'role': 'system', 'content': SYSTEM})

## 5. 모델 불러오기 + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, dtype=None)
# 비전 모델로 불러지면 tokenizer가 processor일 수 있음 → 글자 처리는 안쪽 tokenizer로
tok = getattr(tokenizer, 'tokenizer', tokenizer)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)
model.print_trainable_parameters()

## 6. 채팅 템플릿 적용
Qwen 템플릿이 도구 정의는 시스템 프롬프트에, 도구 호출은 `<tool_call>`, 결과는 `<tool_response>`로 넣는다.
thinking은 끈다 (서버에서도 끄고 쓸 것).

In [ ]:
from datasets import Dataset

def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(messages, tools=TOOLS, tokenize=False,
                                   add_generation_prompt=add_generation_prompt, enable_thinking=False)

texts = [render(r['messages']) for r in rows]
lens = [len(tok(t).input_ids) for t in texts]
keep = [t for t, n in zip(texts, lens) if n <= MAX_SEQ_LEN]
print(f'토큰 길이 최대 {max(lens)}, 평균 {sum(lens) // len(lens)} / {MAX_SEQ_LEN} 넘는 {len(texts) - len(keep)}개 제외')
dataset = Dataset.from_dict({'text': keep})
print(keep[0][-1500:])

## 7. 학습
답변(assistant) 부분만 loss를 계산한다. 사용자 질문·도구 결과(`<tool_response>`, user 차례로 들어감)는 제외.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text', output_dir=str(OUT_DIR / 'checkpoints'),
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.05,
        optim='adamw_8bit', weight_decay=0.01, bf16=BF16, fp16=not BF16,
        logging_steps=5, save_strategy='epoch', save_total_limit=1, seed=3407, report_to='none'),
)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f}분, 최대 VRAM {torch.cuda.max_memory_reserved() / 2**30:.1f}GB")

## 8. 확인: 도구를 제대로 부르나
첫 응답에서 `<tool_call>`을 내는지만 본다. 실제 도구 실행까지는 서버(`llm/agent/loop.py`)에 붙여서 확인.

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, fmt='doubles', party=None):
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': ctx(fmt, party) + '\n\n' + question}]
    ids = tok(render(msgs, add_generation_prompt=True), return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=512, temperature=0.7, top_p=0.8, do_sample=True)
    text = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)
    calls = [json.loads(c) for c in re.findall(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, re.S)]
    return text, calls

for q in ['한카리아스 더블에서 보통 어떻게 키워?', '요즘 더블 픽률 1위가 뭐야?', 'SP 합계는 몇이야?']:
    text, calls = ask(q)
    print('Q:', q)
    print('도구 호출:', calls or '없음')
    print(text[:400], '\n' + '-' * 60)

## 9. 저장
LoRA 어댑터만 저장한다 (수십 MB). 서버에서는 원본 모델 + 어댑터로 불러 쓴다.

In [ ]:
ADAPTER = OUT_DIR / 'qwen35-4b-coach-lora'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print('저장:', ADAPTER)

### (선택) 병합·GGUF
llama.cpp/Ollama로 돌리려면 4bit GGUF로 내보낸다 (추론 약 3GB). 시간이 오래 걸리니 필요할 때만.
```python
model.save_pretrained_gguf(str(OUT_DIR / 'gguf'), tokenizer, quantization_method='q4_k_m')
```